# 🚗 Entrenar: SOLO MARCA Vehicular
## Para usar con nuestro Vehicle Vision API

Este notebook entrena un clasificador de **marca** usando YOLOv8cls.

### Estrategia:
- Stanford Cars (196 clases modelo) → agrupamos por **marca** → ~49 marcas
- Más imágenes por marca = más preciso
- Reconoce Toyota, Kia, Hyundai aunque sean modelos distintos

**⚠️ Antes de empezar:**
- Entorno de ejecución → Cambiar tipo → **GPU** ✅
- Ejecuta las celdas **UNA POR UNA**
- La celda 4 (entrenar) tarda ~30-40 min

---

In [ ]:
# CELDA 1: Montar Google Drive + instalar
from google.colab import drive
drive.mount('/content/drive')
!pip install ultralytics datasets -q

In [ ]:
# CELDA 2: Descargar dataset
from datasets import load_dataset
print("Descargando Stanford Cars (~2GB)...")
dataset = load_dataset("tanganke/stanford_cars", split="train")
test_dataset = load_dataset("tanganke/stanford_cars", split="test")
print(f"✅ Train: {len(dataset)} | Test: {len(test_dataset)}")

In [ ]:
# CELDA 3: Agrupar por MARCA (no por modelo)
import os, re
from tqdm import tqdm
from collections import defaultdict

# Obtener nombres de clase
class_names = dataset.features['label'].names  # ej: "Toyota Camry Sedan 2012"

# Extraer marca = primera palabra del nombre
def extract_brand(class_name):
    # Casos especiales:
    name = class_name.split()[0]
    # Mercedes-Benz, Land Rover, etc. vienen completos
    if class_name.startswith("Mercedes"): return "Mercedes-Benz"
    if class_name.startswith("Land"): return "Land Rover"
    if class_name.startswith("Rolls"): return "Rolls-Royce"
    if class_name.startswith("Alfa"): return "Alfa Romeo"
    if class_name.startswith("Aston"): return "Aston Martin"
    return name

# Mapa: label_id → marca
brand_map = {i: extract_brand(name) for i, name in enumerate(class_names)}
brands = sorted(set(brand_map.values()))
print(f"📊 {len(class_names)} modelos → {len(brands)} marcas")
print("Marcas:", ', '.join(brands))

# Guardar train agrupado por marca
base = "/content/stanford_cars_yolo/train"
for i, item in enumerate(tqdm(dataset)):
    marca = brand_map[item['label']]
    marca_dir = f"{base}/{marca}"
    os.makedirs(marca_dir, exist_ok=True)
    item['image'].save(f"{marca_dir}/img_{i:05d}.jpg")

# Guardar test agrupado por marca
base_test = "/content/stanford_cars_yolo/test"
for i, item in enumerate(tqdm(test_dataset)):
    marca = brand_map[item['label']]
    marca_dir = f"{base_test}/{marca}"
    os.makedirs(marca_dir, exist_ok=True)
    item['image'].save(f"{marca_dir}/img_test_{i:05d}.jpg")

# Mostrar distribución
from collections import Counter
counts = Counter(brand_map[item['label']] for item in dataset)
print("\n📊 Distribución por marca (top 15):")
for marca, n in counts.most_common(15):
    print(f"   {marca}: {n} fotos")
print(f"\n✅ Total: {len(dataset)} train + {len(test_dataset)} test en {len(brands)} marcas")

In [ ]:
# CELDA 4: ENTRENAR (solo marca, ~30-40 min con GPU)
from ultralytics import YOLO

model = YOLO('yolov8n-cls.pt')  # nano, ~6MB

results = model.train(
    data='/content/stanford_cars_yolo',
    epochs=30,           # menos épocas porque son menos clases
    imgsz=224,
    batch=32,
    device='cuda',
    workers=4,
    lr0=0.001,
    patience=5,
    name='marca_solo',
    project='/content/drive/MyDrive/vehicle_ai'
)
print("✅ Entrenamiento completado")

In [ ]:
# CELDA 5: Exportar modelo ONNX
model.export(format='onnx', imgsz=224)

print("✅ Modelo exportado a ONNX")
print("📁 /content/drive/MyDrive/vehicle_ai/marca_solo/weights/best.onnx")
print()
print("📥 Para subir al VPS:")
print("   scp best.onnx root@2.25.128.81:/opt/docker-projects/")
print("   plate_recognition/vehicle_vision/models/marca_modelo.onnx")

### 📝 Notas

**¿Por qué solo marca?**
- ~49 marcas vs 196 modelos: más imágenes por clase = más preciso
- Cubre autos + SUVs + pickups de cada marca
- Más rápido de entrenar (30 min vs 2h)

**Después del entrenamiento:**
1. Descarga `best.onnx` de tu Drive
2. Súbelo al VPS (te ayudo con eso)
3. Activo el endpoint en el API
4. Listo para probar con fotos reales